# 02 · Contrastive refinement (stage C), two runs
CR against the cached teacher document vectors (a frozen lookup table, no document encoder).
- **Run A**: from the original pretrained student → `cr_only/`
- **Run B**: from the aligned checkpoint → `aligned_cr/` (needs 01 first)

The two runs differ only in `init` and `out_dir`. Run with `DEBUG = True` first.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch

DEBUG = True    # first pass: ~10k passages, Qwen3-Embedding-0.6B as debug teacher, 3 steps per loop, shape printing
TINY = False    # local CPU smoke run: tiny random models + fake rows, no downloads
NPROCS = torch.cuda.device_count() or 2   # DDP processes (2 CPU processes with gloo when there is no GPU)

# --- paths: each mode gets its own cache + runs dirs, so a debug pass never overwrites a real run
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "tiny" if TINY else "debug" if DEBUG else "full"
PATHS = dict(cache_dir=f"{ROOT}/cache/{MODE}", runs_dir=f"{ROOT}/runs/{MODE}")
PATHS.update(data_path=f"{PATHS['cache_dir']}/data.json",       # built by 00, cell 1
             emb_dir=f"{PATHS['cache_dir']}/emb/teacher")         # built by 00, cell 2

# --- models
MODELS = dict(
    teacher="tiny:1" if TINY else "Qwen/Qwen3-Embedding-0.6B" if DEBUG else "Qwen/Qwen3-Embedding-4B",
    student="tiny:0" if TINY else "Qwen/Qwen3-Embedding-0.6B",
)
DIM = 32 if TINY else 256   # MRL dim shared by teacher index and student (tiny model has hidden 64)

CR_CFG = dict(
    # model
    dim=DIM, max_query_len=32,
    trainable="all_but_embeddings",
    # loss (see losses.py)
    loss="contrastive",
    loss_kwargs=dict(tau=0.05,          # temperature
                     n_hard_neg=1,      # 0 or 1: the corpus keeps one stored negative per query
                     same_tower=True,   # other queries in the batch as negatives
                     mask_margin=0.1,   # mask negatives scoring > positive + margin (likely false negatives)
                     extra_negs=0),     # extra random negatives drawn from the corpus table per step
    # optimisation
    lr=1e-5, lr_schedule="cosine", warmup_ratio=0.05, min_lr_ratio=0.1,
    epochs=1, batch_size=128, accum_steps=1, max_grad_norm=1.0,
    amp=True,
    # run
    seed=0, val_every=200, save_dtype="fp16",
    debug=DEBUG,
    # io
    data_path=PATHS["data_path"], emb_dir=PATHS["emb_dir"],
)
if TINY:
    CR_CFG.update(batch_size=4, max_query_len=16, amp=False)

RUN_A = dict(init=MODELS["student"],                  out_dir=f"{PATHS['runs_dir']}/cr_only")
RUN_B = dict(init=f"{PATHS['runs_dir']}/aligned",     out_dir=f"{PATHS['runs_dir']}/aligned_cr")

## Plan (Run A)

In [ ]:
from ddp import launch
from train import cr_main

launch(cr_main, {**CR_CFG, **RUN_A, "plan_only": True}, nprocs=NPROCS)

## Debug run
3 steps per GPU, tensor shapes, grad check, weights-in-sync check. Saves to a separate dir.

In [ ]:
launch(cr_main, {**CR_CFG, **RUN_A, "debug": True, "out_dir": f"{PATHS['runs_dir']}/debug_cr"}, nprocs=NPROCS)

## Run A: CR from the pretrained student

In [ ]:
launch(cr_main, {**CR_CFG, **RUN_A}, nprocs=NPROCS)

## Run B: CR from the aligned checkpoint

In [ ]:
if not os.path.exists(f"{RUN_B['init']}/config.json"):
    raise FileNotFoundError(f"{RUN_B['init']} not found: run 01_align_only.ipynb (real run) first")
launch(cr_main, {**CR_CFG, **RUN_B}, nprocs=NPROCS)

In [ ]:
# what was saved
for out in (RUN_A["out_dir"], RUN_B["out_dir"],):
    print(out)
    for f in sorted(os.listdir(out)):
        print(f"  {os.path.getsize(os.path.join(out, f)) / 1e6:10.2f} MB  {f}")
    with open(os.path.join(out, "train_log.jsonl")) as fh:
        print("  last log:", fh.readlines()[-1].strip())
print("Next: re-run the last cell of 00_baseline.ipynb to evaluate these checkpoints.")